# R02. Stack Shares
- This calculates the shares of each stack in player results
- Type: Research
- Run Frequency: Irregular
- Sources:
    - A01. DraftKings
- Created: 3/2/2025
- Updated: 3/2/2025

### Imports

In [ ]:
from DataImports import *

### Settings

In [ ]:
start_date, end_date = "20240101", "20241231"  # Date range for game simulations

### Games

In [ ]:
all_game_df = pd.read_csv(os.path.join(baseball_path, "game_df.csv"))

In [ ]:
game_df = all_game_df[(all_game_df['date'].astype(str) >= start_date) & (all_game_df['date'].astype(str) <= end_date)].reset_index(drop=True)

### Contests

Read in contests

In [ ]:
contest_df = create_contests(start_date=start_date, end_date=end_date, name="Relay", entryFee=None, exclusions=['vs', 'Turbo', '@'])

contest_df = contest_df.drop(columns=['game_type', 'game_num', 'date', 'away_score', 'home_score']).merge(game_df, on=['game_id'], how='left')
# contest_df.sort_values('game_datetime').drop_duplicates('contestKey', keep='first').reset_index(drop=True).head(1)

Select contestKeys

In [ ]:
selections = []  # List of contestKeys for contests to run (leave empty for all contests)

# If none are selected
if selections == []:
    # Run all
    selections = contest_df['contestKey'].astype(int).unique()

# Subset contest_df to include only contests of interest
selected_contest_df = contest_df[contest_df['contestKey'].astype(int).isin(selections)].reset_index(drop=True)

# Sort
selected_contest_df.sort_values('game_datetime', ascending=True, inplace=True, ignore_index=True)

### Field Stacks

In [ ]:
def evaluate_contest_stacks(contestKey, draftGroupId, entries, slate_size):
    try:
        standings_df = pd.read_csv(os.path.join(baseball_path, "A09. DraftKings", "5. Entry Results", f"Entry Results {contestKey}.csv"), encoding='iso-8859-1')
        draftable_df = pd.read_csv(os.path.join(baseball_path, "A09. DraftKings", "2. Draftables", f"Draftables {draftGroupId}.csv"), encoding='iso-8859-1', usecols=['Name', 'TeamAbbrev', 'Roster Position', 'Salary'])
        player_results_df = pd.read_csv(os.path.join(baseball_path, "A09. DraftKings", "6. Player Results", f"Player Results {contestKey}.csv"), encoding='iso-8859-1', usecols=['Player', '%Drafted'])
    except:
        return

    player_results_df['%Drafted'] = player_results_df['%Drafted'].str.replace('%', '').astype(float)


    # Merge player teams and salary onto standings
    for position in ['P', 'P.1']:
        standings_df = standings_df.merge(
            draftable_df.query('`Roster Position` == "P"').rename(columns={'Salary': f'${position}'}),
            left_on=[position], right_on=['Name'], how='left', suffixes=("", f"_{position}")
        )
    for position in ['C', '1B', '2B', '3B', 'SS', 'OF', 'OF.1', 'OF.2']:
        standings_df = standings_df.merge(
            draftable_df.query('`Roster Position` != "P"').drop_duplicates('Name', keep='first').rename(columns={'Salary': f'${position}'}),
            left_on=[position], right_on=['Name'], how='left', suffixes=("", f"_{position}")
        )

    # Drop Name and Roster Position columns from draftable merges
    standings_df = standings_df.drop(columns=standings_df.filter(regex='^Name').columns)
    standings_df = standings_df.drop(columns=standings_df.filter(like='Roster Position').columns)

    # Merge actual ownership for each position slot
    player_results_df = player_results_df.drop_duplicates('Player', keep='first')
    for position in ['P', 'P.1', 'C', '1B', '2B', '3B', 'SS', 'OF', 'OF.1', 'OF.2']:
        standings_df = standings_df.merge(
            player_results_df.rename(columns={'%Drafted': f'{position}%'}),
            left_on=[position],
            right_on=['Player'],
            how='left'
        )
        standings_df = standings_df.drop(columns=['Player'])

    # Determine if any players are locked or missing
    standings_df.fillna("MISSING", inplace=True)
    standings_df['Invalid Lineup'] = standings_df[
        ['P', 'P.1', 'C', '1B', '2B', '3B', 'SS', 'OF', 'OF.1', 'OF.2'] +
        [col for col in standings_df.columns if "TeamAbbrev" in col]
    ].isin(['LOCKED', 'MISSING']).any(axis=1).astype(int)

    # Determine stack
    hitter_team_cols = ['TeamAbbrev_C', 'TeamAbbrev_1B', 'TeamAbbrev_2B', 'TeamAbbrev_3B',
                        'TeamAbbrev_SS', 'TeamAbbrev_OF', 'TeamAbbrev_OF.1', 'TeamAbbrev_OF.2']

    def determine_stack(row):
        counts = row.value_counts().sort_values(ascending=False)
        return '-'.join(map(str, counts.values))

    standings_df['Stack'] = standings_df[hitter_team_cols].apply(determine_stack, axis=1)

    # Pitcher salary tiers (relative to this slate)
    pitcher_salaries = draftable_df.query('`Roster Position` == "P"')['Salary'].dropna()
    p25, p75 = pitcher_salaries.quantile(0.25), pitcher_salaries.quantile(0.75)

    salary_cols = ['$P', '$P.1', '$C', '$1B', '$2B', '$3B', '$SS', '$OF', '$OF.1', '$OF.2']
    hitter_salary_cols = ['$C', '$1B', '$2B', '$3B', '$SS', '$OF', '$OF.1', '$OF.2']

    # Use Float64 (nullable) to handle MISSING values cleanly
    standings_df[salary_cols] = standings_df[salary_cols].replace('MISSING', pd.NA).astype('Float64')

    def pitcher_tier(salary):
        if pd.isna(salary): return 'MISSING'
        if salary >= p75: return 'expensive'
        if salary <= p25: return 'cheap'
        return 'mid'

    for pos in ['P', 'P.1']:
        standings_df[f'${pos}_Tier'] = standings_df[f'${pos}'].apply(pitcher_tier)

    standings_df['P_Tier'] = standings_df[['$P_Tier', '$P.1_Tier']].apply(
        lambda r: '/'.join(sorted(r.fillna('MISSING').values)), axis=1
    )

    standings_df['P_Salary_Total'] = standings_df[['$P', '$P.1']].sum(axis=1)
    standings_df['Hitter_Salary_Total'] = standings_df[hitter_salary_cols].sum(axis=1)
    standings_df['Hitter_Salary_Avg'] = standings_df[hitter_salary_cols].mean(axis=1)
    standings_df['Total_Salary'] = standings_df['P_Salary_Total'] + standings_df['Hitter_Salary_Total']

    # Evaluate performance
    n = len(standings_df)
    standings_df['Percentile'] = 1 - (standings_df['Rank'] - 1) / n
    standings_df['Top1pct'] = (standings_df['Rank'] <= n * 0.01).astype(int)
    standings_df['Top5pct'] = (standings_df['Rank'] <= n * 0.05).astype(int)

    # Add contest information
    standings_df['contestKey'] = contestKey
    standings_df['draftGroupId'] = draftGroupId
    standings_df['entries'] = entries
    standings_df['slate_size'] = slate_size

    # Flag any contest where any lineup has a locked/missing player
    standings_df['Invalid Contest'] = standings_df.groupby('contestKey')['Invalid Lineup'].transform('max')

    return standings_df


def build_stack_lookup(standings_df):
    df = standings_df[standings_df['Invalid Lineup'] == 0].copy()
    total = len(df)

    summary = (
        df.groupby(['Stack', 'P_Tier'])
        .agg(
            count=('Percentile', 'size'),
            avg_percentile=('Percentile', 'mean'),
            top1pct_rate=('Top1pct', 'mean'),
            top5pct_rate=('Top5pct', 'mean'),
            avg_hitter_salary=('Hitter_Salary_Avg', 'mean'),
            p25_hitter_salary=('Hitter_Salary_Avg', lambda x: x.quantile(0.25)),
            p75_hitter_salary=('Hitter_Salary_Avg', lambda x: x.quantile(0.75)),
            avg_pitcher_salary=('P_Salary_Total', 'mean'),
            avg_total_salary=('Total_Salary', 'mean'),
        )
        .reset_index()
    )

    summary['field_pct'] = summary['count'] / total
    return summary.sort_values('count', ascending=False)

In [ ]:
results = Parallel(n_jobs=-1)(
    delayed(evaluate_contest_stacks)(
        r.contestKey,
        r.draftGroupId,
        r.entries,
        r.slate_size
    )
    for r in tqdm(selected_contest_df.drop_duplicates(subset='draftGroupId').itertuples(index=False), total=len(selected_contest_df['contestKey'].unique()))
)
standings_df = pd.concat(results, ignore_index=True)

# Then to get the lookup table:
lookup = build_stack_lookup(standings_df)

Remove Invalid Contests

These are caused by entries with "LOCKED" players or missing team information

In [ ]:
standings_df = standings_df[standings_df['Invalid Contest'] == 0]

Remove Invalid Stacks 

Rare: Can be caused by multiple position players sharing a name

In [ ]:
standings_df = standings_df[~standings_df['Stack'].str.contains("6")]

### Evaluations

##### Stack Performance

All

In [ ]:
standings_df.groupby('Stack')[['Percentile', 'Top1pct', 'Top5pct']].mean().sort_values('Top5pct', ascending=False)

Small Slates

In [ ]:
standings_df.query('slate_size < 8').groupby('Stack')[['Percentile', 'Top1pct', 'Top5pct']].mean().sort_values('Top5pct', ascending=False)

Large Slates

In [ ]:
standings_df.query('slate_size >= 8').groupby('Stack')[['Percentile', 'Top1pct', 'Top5pct']].mean().sort_values('Top5pct', ascending=False)

##### Stack Shares

All

In [ ]:
standings_df['Stack'].value_counts(normalize=True)

Top 5

In [ ]:
top5_stack_share = (standings_df['Stack'].value_counts().head(5))
top5_stack_share / top5_stack_share.sum()

Small Slates

In [ ]:
top5_stack_share = (standings_df.query('slate_size < 8')['Stack'].value_counts().head(5))
top5_stack_share / top5_stack_share.sum()

Large Slates

In [ ]:
top5_stack_share = (standings_df.query('slate_size >= 8')['Stack'].value_counts().head(5))
top5_stack_share / top5_stack_share.sum()

In [ ]:
# standings_df.groupby('EntryName')[['Points']].agg({'mean', 'count'}).sort_values('Points', ascending=False).head()

In [ ]:
# Clean username
standings_df['Username'] = (
    standings_df['EntryName']
    .str.replace(r'\(.*?\)', '', regex=True)
    .str.strip()
)

# Contest sizes
standings_df['ContestSize'] = standings_df.groupby('contestKey')['Rank'].transform('size')

# Percentile cutoffs
standings_df['Top1'] = standings_df['Rank'] <= (standings_df['ContestSize'] * 0.01)
standings_df['Top5'] = standings_df['Rank'] <= (standings_df['ContestSize'] * 0.05)

# Aggregate by username
summary = (
    standings_df
    .groupby('Username')
    .agg(
        Entries=('Rank', 'size'),
        Top1Pct=('Top1', 'sum'),
        Top5Pct=('Top5', 'sum'),
        Points=('Points', 'mean')
    )
)

# Rates (optional but usually helpful)
summary['Top1% Rate'] = summary['Top1Pct'] / summary['Entries']
summary['Top5% Rate'] = summary['Top5Pct'] / summary['Entries']

summary = summary.query('Entries > 2000').sort_values('Top1% Rate', ascending=False)

print(len(summary))
summary.head(25)

In [ ]:
summary.describe()

Notes:
- There are five stack types that average a finish in the top 5% more than 5% of the time. These seem like reasonable candidates for inclusion in my optimized lineups
- 4-2-1-1 is a top 5 lineup by usage but not performance